# Notebook 5 - Predictive Maintenance: one recipe, four labs

Learn to predict when a machine will fail, using the same 11-step linear-regression recipe every time.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-05-predictive-maintenance.ipynb)

- Lecture: [Chapter 5 - Predictive Maintenance with Linear Models](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-05-predictive-maintainance.md)
- Labs: [LM501](https://rathachai.github.io/DA-LAB/learnings/linear/lm501.html) | [LM502](https://rathachai.github.io/DA-LAB/learnings/linear/lm502.html) | [LM503](https://rathachai.github.io/DA-LAB/learnings/linear/lm503.html) | [LM504](https://rathachai.github.io/DA-LAB/learnings/linear/lm504.html)

**The recipe:** 1 Load libraries, 2 Get data, 3 Display data, 4 Visualize correlation, 5 Process data (only if needed), 6 Select X and y, 7 Split into train and test, 8 Create and train the model, 9 Predict, 10 Evaluate, 11 Visualize y and y_pred.

**The parts:** A = RUL from sensors (all 11 steps) | B = degradation and threshold | C = early warning | D = cost of decisions. Parts B-D reuse the recipe in a shorter form and say which steps they skip.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

# PART A - Remaining Useful Life (RUL) from sensors

**RUL** is the number of hours a machine still has before it fails. We have 60 machines, 40 snapshots each. `s1`..`s7` are sensors and `y` is the RUL. Goal: estimate `y` from the sensors.

## Step 1 · Load libraries
We use pandas (tables), numpy (maths), scikit-learn (the model) and matplotlib (plots only).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

pd.set_option("display.precision", 2)
plt.rcParams["figure.figsize"] = (8, 3.5)

print("Libraries ready")

## Step 2 · Get data
Read the CSV file straight from the web into a table called `df`.

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm501_data.csv"
df = pd.read_csv(url)
print("loaded", len(df), "rows")

## Step 3 · Display data
Look before you model: first rows, size, and basic statistics. Notice the sensors have very different scales.

In [ ]:
print(df.shape)

In [ ]:
df

In [ ]:
df.describe().T

## Step 4 · Visualize correlation
The correlation `r` runs from -1 to +1. Near +1 or -1 means the sensor moves in a straight line with RUL (useful). Near 0 means no straight-line relation (noise, or a curve).

In [ ]:
sensors = ["s1", "s2", "s3", "s4", "s5", "s6", "s7"]
r = df[sensors].corrwith(df["y"])
r.plot.bar(color="tab:blue"); plt.axhline(0, color="k", lw=0.8)
plt.ylabel("r with y (RUL)"); plt.show()

In [ ]:
print(r.round(2).to_dict())

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 3))
for ax, s in zip(axs, ["s1", "s4", "s7"]):
    ax.scatter(df[s], df["y"], s=6); ax.set_xlabel(s); ax.set_title(f"r = {r[s]:.2f}")
axs[0].set_ylabel("RUL y (h)"); plt.tight_layout(); plt.show()

One machine over time. We rescale every column to 0-1 so they fit on one plot. The thick black line is the RUL: it falls as the machine wears.

In [ ]:
m1 = df[df.machine == 1][sensors + ["y"]]
norm = (m1 - m1.min()) / (m1.max() - m1.min())
ax = norm[sensors].plot(lw=1, figsize=(8, 3.5))
norm["y"].plot(ax=ax, color="k", lw=3, label="RUL (y)")
ax.set_xlabel("snapshot"); ax.set_ylabel("normalised"); ax.legend(ncol=4, fontsize=8); plt.show()

## Step 5 · Process data (only if needed)
`s7` bends upward like an exponential curve, and a straight-line model dislikes curves. The natural log straightens it. We add a new column `ln_s7`.

In [ ]:
df["ln_s7"] = np.log(df["s7"])
print("r(s7, y)    =", round(df["s7"].corr(df["y"]), 3))

In [ ]:
print("r(ln_s7, y) =", round(df["ln_s7"].corr(df["y"]), 3))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].scatter(df["s7"], df["y"], s=6)
axs[0].set_xlabel("s7"); axs[0].set_ylabel("RUL y (h)"); axs[0].set_title(f"s7 (curved), r = {df['s7'].corr(df['y']):.3f}")
axs[1].scatter(df["ln_s7"], df["y"], s=6, color="tab:green")
axs[1].set_xlabel("ln_s7 = np.log(s7)"); axs[1].set_ylabel("RUL y (h)"); axs[1].set_title(f"ln_s7 (straighter), r = {df['ln_s7'].corr(df['y']):.3f}")
plt.tight_layout(); plt.show()

Left: `s7` against `y` bends like a curve, so a straight line fits it poorly. Right: after the log, the cloud lines up much better along a straight line, and `r` is higher.

In [ ]:
df[["s7", "ln_s7", "y"]].head()

## Step 6 · Select X and y
`X` = the inputs (the useful sensors), `y` = what we want to predict. We drop `s3` and `s5` (weak correlation) and use `ln_s7` instead of `s7`.

In [ ]:
features = ["s1", "s2", "s4", "s6", "ln_s7"]
X = df[features]
y = df["y"]
print(X.shape, y.shape)

In [ ]:
X

## Step 7 · Split into train and test
We split **by machine**, not by row. If random rows are used, the same machine appears in both train and test, so the model has "seen" that machine before. In real life the model must work on **new machines**, so whole machines are held out (70% train, 30% test).

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=0)
train_idx, test_idx = next(gss.split(X, y, groups=df["machine"]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print("train machines:", sorted(df["machine"].iloc[train_idx].unique()))

In [ ]:
print("test machines :", sorted(df["machine"].iloc[test_idx].unique()))

In [ ]:
print("train rows", len(X_train), "| test rows", len(X_test))

## Step 8 · Create and train the model
`LinearRegression` finds the best weights so that `y ≈ b0 + b1*s1 + b2*s2 + ...`. `fit` is the learning step and uses the training data only.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
print("intercept b0:", round(model.intercept_, 1))

In [ ]:
pd.Series(model.coef_, index=features).round(2)

## Step 9 · Predict (y_pred)
Give the model the test sensors it has never seen and let it guess the RUL.

In [ ]:
y_pred = model.predict(X_test)
pd.DataFrame({"actual": y_test.values[:6], "predicted": y_pred[:6]}).round(1)

## Step 10 · Evaluate
- **MAE**: average error in hours. **RMSE**: like MAE but punishes big misses.
- **MAPE**: error as a percentage of the true value. **R²**: 1 is perfect, 0 is no better than guessing the average.

In [ ]:
# metrics for the train set and the test set
sets = {"train": (y_train, model.predict(X_train)),
        "test": (y_test, y_pred)}

for label, (y_true, y_hat) in sets.items():
    print("-----", label, "-----")

    mae = mean_absolute_error(y_true, y_hat)
    print("MAE  =", round(mae, 3))

    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    print("RMSE =", round(rmse, 3))

    mape = 100 * mean_absolute_percentage_error(y_true, y_hat)
    print("MAPE =", round(mape, 2), "%")

    r2 = r2_score(y_true, y_hat)
    print("R2   =", round(r2, 3))
    print()

Honest note: for these data the train and test scores are close, so splitting by row or by machine gives almost the same answer here. The by-machine split is still the safe habit, because real machines often have their own "fingerprint" and the gap can then be large.

## Step 11 · Visualize y and y_pred
Points on the dashed diagonal are perfect predictions. Circles are training machines, diamonds are test machines.

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_train, model.predict(X_train), marker="o", s=10, alpha=0.4, label="train")
plt.scatter(y_test, y_pred, marker="D", s=14, alpha=0.7, label="test")
lim = [0, y.max()]; plt.plot(lim, lim, "k--")
plt.xlabel("actual RUL (h)"); plt.ylabel("predicted RUL (h)"); plt.legend(); plt.show()

# PART B - Degradation and threshold (LM502)

A **health index** starts at 100 (new) and falls to 0 (failed). We fit a straight line to the most recent readings and extend it forward to a **repair threshold θ** and to **zero health** (failure).

Recipe steps used: 1 (done), 2, 3, 4, 6, 8, 9, 10, 11. Skipped: 5 (no processing needed) and 7 (we train on the recent window and then look into the future, so the "test" is the real failure time).

In [ ]:
deg = pd.read_csv("https://rathachai.github.io/DA-LAB/learnings/linear/data/lm502_degradation.csv")
print(deg.shape)

In [ ]:
print(deg.groupby("pattern")["failure_time"].first().to_dict())

In [ ]:
deg

**Step 4 here:** true health (grey), sensor reading (blue) and the real failure time (red dashed) for each wear pattern.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(11, 5), sharey=True)
for ax, p in zip(axs.flat, deg["pattern"].unique()):
    s = deg[deg.pattern == p]
    ax.plot(s.t, s.health_sensor, ".", ms=2); ax.plot(s.t, s.health_true, color="gray")
    ax.axvline(s.failure_time.iloc[0], color="r", ls="--"); ax.set_title(p)
axs.flat[-1].axis("off"); plt.tight_layout(); plt.show()

**Steps 6-8.** Pick the `accelerating` pattern and a "now" of 170 h. Use only the **last 60 readings**: X = time `t`, y = `health_sensor`.

In [ ]:
THETA, L, NOW = 50, 60, 170
acc = deg[deg.pattern == "accelerating"]
win = acc[(acc.t <= NOW) & (acc.t > NOW - L)]
X = win[["t"]]
y = win["health_sensor"]
model = LinearRegression().fit(X, y)
print("slope (health points per hour):", round(model.coef_[0], 3))

**Steps 9-10.** Predict health on the window and check the fit.

In [ ]:
y_pred = model.predict(X)

mae = mean_absolute_error(y, y_pred)
print("MAE  =", round(mae, 3))

rmse = np.sqrt(mean_squared_error(y, y_pred))
print("RMSE =", round(rmse, 3))

mape = 100 * mean_absolute_percentage_error(y, y_pred)
print("MAPE =", round(mape, 2), "%")

r2 = r2_score(y, y_pred)
print("R2   =", round(r2, 3))

**Extrapolate.** From the line `health = b0 + b1·t`: repair time = (θ - b0)/b1, failure time = (0 - b0)/b1, and RUL = failure time - now.

In [ ]:
b0, b1 = model.intercept_, model.coef_[0]
t_theta, t_fail = (THETA - b0) / b1, (0 - b0) / b1
true_fail = acc.failure_time.iloc[0]
print(f"predicted repair time (health={THETA}): {t_theta:.0f} h")

In [ ]:
print(f"predicted failure {t_fail:.0f} h | true failure {true_fail} h")

In [ ]:
print(f"predicted RUL {t_fail - NOW:.0f} h | true RUL {true_fail - NOW} h | RUL error {t_fail - true_fail:+.0f} h")

**Step 11.** The red line is the fitted trend, extended to the threshold (orange) and to zero. Wear is accelerating, so the straight line is too optimistic: it crosses zero later than the machine really fails.

In [ ]:
tt = np.linspace(NOW - L, t_fail + 10, 100)
plt.plot(acc.t, acc.health_sensor, ".", ms=3); plt.plot(win.t, win.health_sensor, ".", color="navy")
plt.plot(tt, b0 + b1 * tt, "r", label="fitted line")
plt.axhline(THETA, color="orange", label="threshold θ"); plt.axhline(0, color="k", lw=0.5)
plt.axvline(NOW, color="g", ls=":", label="now"); plt.axvline(true_fail, color="gray", ls="--", label="true failure")
plt.ylim(-10, 105); plt.xlabel("t (h)"); plt.ylabel("health"); plt.legend(fontsize=8); plt.show()

**Same recipe for all five patterns.** A positive RUL error means the line promises more life than the machine really has.

In [ ]:
rows = {}
for p in deg["pattern"].unique():
    s = deg[deg.pattern == p]
    w = s[(s.t <= NOW) & (s.t > NOW - L)]
    m = LinearRegression().fit(w[["t"]], w["health_sensor"])
    tf = -m.intercept_ / m.coef_[0]
    true = s.failure_time.iloc[0]
    rows[p] = {"predicted failure": tf, "true failure": true, "RUL error (h)": tf - true}

pd.DataFrame(rows).T.round(0)

The error is small for the steady `linear` pattern and larger for patterns that change speed (accelerating, late onset, shocks). A line only knows the recent trend, not what will happen next.

# PART C - Early warning from residuals (LM503)

Idea: learn what a **healthy** signal looks like, predict the next value, and watch the **residual** = actual - predicted. When the residual becomes unusually big, something has changed, like a point outside the limits of a control chart.

Recipe steps used: 2, 3, 4, 5 (build the lag table), 6, 7 (first 180 points = train, the rest = test), 8, 9, 10, 11. An **AR(5)** model predicts each value from the previous 5 values, using plain `LinearRegression`.

In [ ]:
sig = pd.read_csv("https://rathachai.github.io/DA-LAB/learnings/linear/data/lm503_signals.csv")
print(sig.scenario.unique())

In [ ]:
sig

**Step 4 here:** each scenario over time. Red dots mark where the fault is really active (`is_fault = 1`).

In [ ]:
scen = list(sig.scenario.unique())
fig, axs = plt.subplots(len(scen), 1, figsize=(9, 8), sharex=True)
for ax, sc in zip(axs, scen):
    s = sig[sig.scenario == sc]
    ax.plot(s.t, s.signal, lw=0.8); ax.plot(s.t[s.is_fault == 1], s.signal[s.is_fault == 1], "r.", ms=3)
    ax.set_ylabel(sc, fontsize=8)
plt.tight_layout(); plt.show()

**Steps 5-8.** Build a sliding-window table: columns `lag1..lag5` (the 5 previous values) and `y` (the current value). Train on the healthy start only (first 180 points) of the `drift` scenario.

In [ ]:
W, NTRAIN = 5, 180
s = sig[sig.scenario == "drift"].set_index("t")

# lag table: lag1..lag5 = the 5 previous values, y = the current value
tab = pd.DataFrame(index=s.index)
for i in range(1, W + 1):
    tab[f"lag{i}"] = s["signal"].shift(i)
tab["y"] = s["signal"]
tab = tab.dropna()

train = tab[tab.index < NTRAIN]
X_train, y_train = train.drop(columns="y"), train["y"]
model = LinearRegression().fit(X_train, y_train)
tab

**Steps 9-10.** Predict every point, compute residuals, and take **σ** (sigma) = the typical size of residuals on the healthy training part. Alarm when `|residual| > k·σ` for 3 samples in a row.

In [ ]:
X, y = tab.drop(columns="y"), tab["y"]
y_pred = pd.Series(model.predict(X), index=tab.index)
resid = y - y_pred
sigma = resid[resid.index < NTRAIN].std()
print("sigma =", round(sigma, 3))

In [ ]:
rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
print("train RMSE =", round(rmse, 3))

In [ ]:
# drift scenario: alarm when |residual| > k*sigma for 3 samples in a row
fs = s.index[s.is_fault == 1].min()
rows = {}
for k in [2, 3]:
    over = (resid.abs() > k * sigma).astype(int)
    alarm = over.rolling(3).sum() == 3
    false = int(alarm[(alarm.index >= NTRAIN) & (alarm.index < fs)].sum())
    hit = alarm[alarm.index >= fs]
    delay = hit.idxmax() - fs if hit.any() else np.nan
    rows[f"k={k}"] = {"delay": delay, "false alarms": false}

pd.DataFrame(rows).T

**Step 11.** Top: signal and AR prediction. Bottom: residuals with the ±2σ limits and the alarms (red). Green dotted = end of training, grey dashed = true fault start. The detection **delay** is the first alarm minus the fault start; **false alarms** are alarm samples while the machine was still healthy.

In [ ]:
K = 2; fs = s.index[s.is_fault == 1].min()
over = (resid.abs() > K * sigma).astype(int)
al = over.rolling(3).sum() == 3
fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axs[0].plot(s.signal, lw=0.8, label="signal"); axs[0].plot(y_pred, lw=0.8, label="AR(5) prediction"); axs[0].legend()
axs[1].plot(resid, lw=0.8); axs[1].axhline(K * sigma, color="orange"); axs[1].axhline(-K * sigma, color="orange")
axs[1].plot(resid[al], "r.", label="alarm"); axs[1].set_ylabel("residual"); axs[1].legend()
for a in axs: a.axvline(NTRAIN, color="g", ls=":"); a.axvline(fs, color="gray", ls="--")
plt.tight_layout(); plt.show()

**All scenarios, k = 2 and k = 3.** `none` has no fault, so only false alarms matter there. NaN delay = never detected.

In [ ]:
rows = []
for sc in scen:
    # one scenario: build lag table, train on healthy start, get residuals
    d = sig[sig.scenario == sc].set_index("t")
    tb = pd.DataFrame(index=d.index)
    for i in range(1, W + 1):
        tb[f"lag{i}"] = d["signal"].shift(i)
    tb["y"] = d["signal"]
    tb = tb.dropna()

    healthy = tb[tb.index < NTRAIN]
    mdl = LinearRegression().fit(healthy.drop(columns="y"), healthy["y"])
    res = tb["y"] - mdl.predict(tb.drop(columns="y"))
    sg = res[res.index < NTRAIN].std()

    if d.is_fault.any():
        fault_start = d.index[d.is_fault == 1].min()
    else:
        fault_start = np.inf

    for k in [2, 3]:
        over = (res.abs() > k * sg).astype(int)
        alarm = over.rolling(3).sum() == 3
        false = int(alarm[(alarm.index >= NTRAIN) & (alarm.index < fault_start)].sum())
        hit = alarm[alarm.index >= fault_start]
        delay = hit.idxmax() - fault_start if hit.any() else np.nan
        rows.append({"scenario": sc, "k": k, "delay": delay, "false alarms": false})

pd.DataFrame(rows).set_index(["scenario", "k"])

Honest reading: a sudden change (level shift, oscillation, variance) is caught within a few samples. A **slow drift** is the hard case, because the AR model partly follows the drift and the residual stays small, so k = 3 may miss it completely. A smaller k catches more but raises the risk of false alarms: you choose the trade-off.

# PART D - Cost of decisions (LM504)

A prediction is only useful if it leads to a better decision, and wrong decisions have **unequal costs**: a breakdown is far more expensive than a planned repair.

Recipe steps used: 2, 3, 10, 11 only. Skipped: 4-9, because the three predictors are already given as columns (`predicted_*`) and we only judge them.

In [ ]:
fleet = pd.read_csv("https://rathachai.github.io/DA-LAB/learnings/linear/data/lm504_fleet.csv")
preds = ["predicted_unbiased", "predicted_pessimistic", "predicted_optimistic"]
print(fleet.shape)

In [ ]:
fleet

**Step 11.** Actual vs predicted for each predictor. Above the dashed line = optimistic (promises too much life), below = pessimistic.

In [ ]:
y = fleet["actual_rul"]
fig, axs = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)
for ax, p in zip(axs, preds):
    ax.scatter(y, fleet[p], s=8); ax.plot([0, y.max()], [0, y.max()], "k--"); ax.set_title(p); ax.set_xlabel("actual RUL (h)")
axs[0].set_ylabel("predicted RUL (h)"); plt.tight_layout(); plt.show()

**Step 10.** Metrics plus **bias** = mean(predicted - actual). Positive bias = optimistic, negative = pessimistic.

In [ ]:
rows = {}
for p in preds:
    print("-----", p, "-----")

    mae = mean_absolute_error(y, fleet[p])
    print("MAE  =", round(mae, 3))

    rmse = np.sqrt(mean_squared_error(y, fleet[p]))
    print("RMSE =", round(rmse, 3))

    mape = 100 * mean_absolute_percentage_error(y, fleet[p])
    print("MAPE =", round(mape, 2), "%")

    r2 = r2_score(y, fleet[p])
    print("R2   =", round(r2, 3))

    bias = (fleet[p] - y).mean()
    print("bias =", round(bias, 3))
    print()

    rows[p] = {"MAE": mae, "RMSE": rmse, "MAPE %": mape, "R2": r2, "bias": bias}

met = pd.DataFrame(rows).T
met

**The decision rule.** If predicted RUL ≤ **H**, repair now: cost = Cp + Cw × actual RUL (the unused life is wasted). Otherwise run until the next inspection D = 50 h later: if the machine's actual RUL is below D it breaks down (cost Cf), else nothing. Costs: Cp = 1000, Cw = 5 per hour, Cf = 8000.

In [ ]:
CP, CW, CF, D = 1000, 5, 8000, 50
H = 100

rows = {}
for p in preds:
    repair = fleet[p] <= H
    cost = np.where(repair, CP + CW * y, np.where(y < D, CF, 0.0)).sum()
    rows[p] = {"repairs": int(repair.sum()),
               "breakdowns": int(((fleet[p] > H) & (y < D)).sum()),
               "total cost": cost}

pd.DataFrame(rows).T

**Cost versus H.** A low H means late repairs and breakdowns; a high H means early repairs and wasted life. The black dot is each predictor's best H.

In [ ]:
Hs = np.arange(0, 301, 5)
curve_data = {}
for p in preds:
    costs = []
    for h in Hs:
        repair = fleet[p] <= h
        costs.append(np.where(repair, CP + CW * y, np.where(y < D, CF, 0.0)).sum())
    curve_data[p] = costs
curves = pd.DataFrame(curve_data, index=Hs)

ax = curves.plot(figsize=(8, 3.8))
for p in preds: ax.plot(curves[p].idxmin(), curves[p].min(), "ko")
ax.set_xlabel("H (h)"); ax.set_ylabel("total cost"); plt.show()

**Accuracy is not the same as cost.** We compare the RMSE ranking with the cost ranking, each predictor at its own best H.

In [ ]:
best = pd.DataFrame({"RMSE": met["RMSE"], "bias": met["bias"], "best H": curves.idxmin(), "min total cost": curves.min()})
best["RMSE rank"] = best["RMSE"].rank().astype(int)
best["cost rank"] = best["min total cost"].rank().astype(int)
print("lowest RMSE:", best["RMSE"].idxmin(), "| cheapest:", best["min total cost"].idxmin())

In [ ]:
best

Read the table honestly: the predictor with the lowest RMSE is not necessarily the cheapest. A cautious (pessimistic) predictor can win on cost because a breakdown costs eight times a repair, so a prediction that errs on the safe side is cheap insurance. Always judge a model by the **cost of the decisions** it drives, not only by its error.

# Summary

- **One recipe:** load, look, visualise, process only if needed, choose X and y, split, fit, predict, evaluate, plot. The same steps work for sensors, trends, signals and costs.
- **Split by machine** to mimic new machines; here the difference from a by-row split is small, but it is the safe habit.
- **Trend and residual methods have limits:** a straight line is optimistic for accelerating wear, and slow drift can slip past a k·σ alarm at k = 3.
- **Judge by cost:** the lowest-RMSE predictor is not always the cheapest, so evaluate the decisions, not just the errors.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io